# N03 — Điền NULL cột creatinine_phosphokinase
**Võ Minh Huy — 2311110117**

CPK là tên viết gọn của creatinine_phosphokinase. Notebook chạy độc lập bằng **Run All**, giữ nguyên CSV before. Các ô đầu dùng lại code chuẩn bị, N01 sex và N02 age đã hoàn thành. Chỉ chụp phần N03 tại mục 2, mục 3 và nhật ký ở mục 4.

## 1. Tạo lại đầu vào sau N02
Đây là code cũ để tạo đúng trạng thái đầu vào; không cần chạy lại notebook trước trong tab khác.

In [1]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
from collections import Counter
import csv
import pandas as pd

thu_muc = Path.cwd()
if thu_muc.name == "python":
    thu_muc = thu_muc.parent
file_csv = thu_muc / "VoMinhHuy_Heart Failure Prediction_before.csv"
with file_csv.open(encoding="utf-8-sig", newline="") as f:
    du_lieu_goc = list(csv.DictReader(f))

def doc_so(gia_tri, cot):
    chuoi = gia_tri.strip()
    if cot == "sex":
        chuoi = {"m":"1", "male":"1", "f":"0", "female":"0"}.get(chuoi.lower(), chuoi)
    try:
        return Decimal(chuoi).quantize(Decimal("0.000001"))
    except InvalidOperation:
        return None

ho_so, da_gap = [], set()
for dong in du_lieu_goc:
    if not any(v.strip() for v in dong.values()):
        continue
    ban_ghi = {cot: doc_so(v, cot) for cot, v in dong.items()}
    khoa = tuple(ban_ghi.values())
    if khoa not in da_gap:
        da_gap.add(khoa)
        ho_so.append(ban_ghi)
df = pd.DataFrame(ho_so)
df.insert(0, "MaHoSo", range(1, len(df)+1))
mau_cho = df.loc[df["sex"].isin([0, 1])].copy()
pd.DataFrame({"SoHoSo":[len(df)], "SoSexThieuTruocDien":[df["sex"].isna().sum()]})

,SoHoSo,SoSexThieuTruocDien
0,299,20


In [2]:
cac_nhom = [
    ["anaemia", "diabetes", "smoking", "high_blood_pressure"],
    ["anaemia", "diabetes"], ["diabetes"], []
]
nhat_ky = []
for i in df.index[df["sex"].isna()]:
    ket_qua = {"MaHoSo":df.at[i, "MaHoSo"],
               **{c:df.at[i,c] for c in cac_nhom[0]},
               "CapNhom":None, "SoMauCho":0, "GiaTriDien":None}
    for cap, cac_cot in enumerate(cac_nhom, start=1):
        nhom = mau_cho
        for cot in cac_cot:
            nhom = nhom.loc[nhom[cot].eq(df.at[i,cot])]
        if len(nhom) < 5:
            continue
        tan_suat = Counter(nhom["sex"])
        mode = min(v for v, n in tan_suat.items() if n == max(tan_suat.values()))
        df.at[i, "sex"] = mode
        ket_qua.update(CapNhom=cap, SoMauCho=len(nhom), GiaTriDien=mode)
        break
    nhat_ky.append(ket_qua)
nhat_ky = pd.DataFrame(nhat_ky)
tong_ket = pd.DataFrame({"SoHoSo":[len(df)],
    "SoSexThieuTruocDien":[len(nhat_ky)],
    "SoODaDien":[nhat_ky["GiaTriDien"].notna().sum()],
    "SoSexThieuSauDien":[df["sex"].isna().sum()]})
tong_ket

,SoHoSo,SoSexThieuTruocDien,SoODaDien,SoSexThieuSauDien
0,299,20,20,0


In [3]:
from decimal import ROUND_HALF_UP
from statistics import median

# df là kết quả sau N01; ho_so giữ dữ liệu quan sát trước N01.
age_work = df.copy(deep=True)
truoc_n01 = pd.DataFrame(ho_so)
tuoi_nhieu = [Decimal("-12"), Decimal("250")]
mat_na_can_dien = age_work["age"].isna() | age_work["age"].isin(tuoi_nhieu)
age_targets = age_work.loc[mat_na_can_dien, ["MaHoSo", "age", "sex", "diabetes"]].copy()
age_donors = truoc_n01.loc[
    truoc_n01["age"].notna() & ~truoc_n01["age"].isin(tuoi_nhieu),
    ["age", "sex", "diabetes"]
].copy()
age_work.loc[age_work["age"].isin(tuoi_nhieu), "age"] = None

pd.DataFrame({"SoHoSo":[len(age_work)],
    "SoNullHoacSaiKieu":[age_targets["age"].isna().sum()],
    "SoTuoiNhieu":[age_targets["age"].isin(tuoi_nhieu).sum()],
    "SoOCanDien":[len(age_targets)]})

,SoHoSo,SoNullHoacSaiKieu,SoTuoiNhieu,SoOCanDien
0,299,21,2,23


In [4]:
cac_nhom_age = [["sex", "diabetes"], ["sex"], []]
dong_nhat_ky_age = []
for i, r in age_targets.iterrows():
    ket_qua_age = {"MaHoSo":r["MaHoSo"], "TuoiTruoc":r["age"],
        "sex":r["sex"], "diabetes":r["diabetes"],
        "LyDo":"Tuổi bị làm nhiễu" if r["age"] in tuoi_nhieu else "NULL hoặc sai kiểu",
        "CapNhom":None, "SoMauCho":0, "MedianNhom":None, "TuoiDien":None}
    for cap, cac_cot in enumerate(cac_nhom_age, start=1):
        nhom = age_donors
        for cot in cac_cot:
            nhom = nhom.loc[nhom[cot].eq(r[cot])]
        if len(nhom) < 5:
            continue
        median_nhom = median(nhom["age"].tolist())
        tuoi_dien = median_nhom.quantize(Decimal("0.1"), rounding=ROUND_HALF_UP)
        age_work.at[i, "age"] = tuoi_dien
        ket_qua_age.update(CapNhom=cap, SoMauCho=len(nhom),
                          MedianNhom=median_nhom, TuoiDien=tuoi_dien)
        break
    dong_nhat_ky_age.append(ket_qua_age)
nhat_ky_age = pd.DataFrame(dong_nhat_ky_age)
tong_ket_age = pd.DataFrame({"SoHoSo":[len(age_work)],
    "SoNullHoacSaiKieu":[age_targets["age"].isna().sum()],
    "SoTuoiNhieu":[age_targets["age"].isin(tuoi_nhieu).sum()],
    "SoOCanDien":[len(age_targets)],
    "SoODaDien":[nhat_ky_age["TuoiDien"].notna().sum()],
    "SoAgeThieuSauDien":[age_work["age"].isna().sum()]})
tong_ket_age

,SoHoSo,SoNullHoacSaiKieu,SoTuoiNhieu,SoOCanDien,SoODaDien,SoAgeThieuSauDien
0,299,21,2,23,23,0


## 2. Chuẩn bị CPK và nhóm tuổi
A1: dưới 50; A2: từ 50 đến dưới 60; A3: từ 60 đến dưới 70; A4: từ 70 trở lên. Hồ sơ nhận dùng tuổi sau N02. Mẫu cho dùng tuổi và CPK quan sát trước mọi bước điền; tuổi thiếu hoặc nhiễu không tạo khóa nhóm, nhưng CPK hợp lệ vẫn dùng ở cấp toàn cột. Có 20 ô thiếu và 1 giá trị nhiễu 99999 cần xử lý.

In [5]:
def nhom_tuoi(tuoi):
    if pd.isna(tuoi) or tuoi in tuoi_nhieu:
        return None
    return "A1" if tuoi < 50 else "A2" if tuoi < 60 else "A3" if tuoi < 70 else "A4"

cot_cpk = "creatinine_phosphokinase"
cpk_nhieu = Decimal("99999")
cpk_work = age_work.copy(deep=True)
cpk_work["age_group"] = cpk_work["age"].map(nhom_tuoi)
mat_na_cpk = cpk_work[cot_cpk].isna() | cpk_work[cot_cpk].eq(cpk_nhieu)
cpk_targets = cpk_work.loc[mat_na_cpk,
    ["MaHoSo", cot_cpk, "age", "age_group", "anaemia"]].copy()
cpk_targets = cpk_targets.rename(columns={cot_cpk:"CPKTruoc"})

# Mẫu cho cố định từ dữ liệu quan sát trước N01, N02.
cpk_donors = truoc_n01.loc[
    truoc_n01[cot_cpk].notna() & ~truoc_n01[cot_cpk].eq(cpk_nhieu),
    [cot_cpk, "age", "anaemia"]].copy()
cpk_donors["age_group"] = cpk_donors["age"].map(nhom_tuoi)
cpk_work.loc[cpk_work[cot_cpk].eq(cpk_nhieu), cot_cpk] = None

pd.DataFrame({"SoHoSo":[len(cpk_work)],
    "SoNullHoacSaiKieu":[cpk_targets["CPKTruoc"].isna().sum()],
    "SoCPKNhieu":[cpk_targets["CPKTruoc"].eq(cpk_nhieu).sum()],
    "SoOCanDien":[len(cpk_targets)]})

,SoHoSo,SoNullHoacSaiKieu,SoCPKNhieu,SoOCanDien
0,299,20,1,21


## 3. Điền median theo nhóm và tổng kết
Thử lần lượt: nhóm tuổi và anaemia → nhóm tuổi → toàn cột. Dùng nhóm đầu có ít nhất 5 mẫu; không đủ ở mọi cấp thì giữ NULL. Chỉ CPK vừa điền được làm tròn số nguyên bằng ROUND_HALF_UP, tương ứng ROUND trong SQL.

In [6]:
cac_nhom_cpk = [["age_group", "anaemia"], ["age_group"], []]
dong_nhat_ky_cpk = []
for i, r in cpk_targets.iterrows():
    ket_qua_cpk = r.to_dict()
    ket_qua_cpk.update(
        LyDo="CPK bị làm nhiễu" if r["CPKTruoc"] == cpk_nhieu else "NULL hoặc sai kiểu",
        CapNhom=None, SoMauCho=0, MedianNhom=None, CPKDien=None)
    for cap, cac_cot in enumerate(cac_nhom_cpk, start=1):
        nhom = cpk_donors
        for cot in cac_cot:
            nhom = nhom.loc[nhom[cot].eq(r[cot])]
        if len(nhom) < 5:
            continue
        median_nhom = median(nhom[cot_cpk].tolist())
        cpk_dien = median_nhom.quantize(Decimal("1"), rounding=ROUND_HALF_UP)
        cpk_work.at[i, cot_cpk] = cpk_dien
        ket_qua_cpk.update(CapNhom=cap, SoMauCho=len(nhom),
                          MedianNhom=median_nhom, CPKDien=cpk_dien)
        break
    dong_nhat_ky_cpk.append(ket_qua_cpk)
nhat_ky_cpk = pd.DataFrame(dong_nhat_ky_cpk)
tong_ket_cpk = pd.DataFrame({"SoHoSo":[len(cpk_work)],
    "SoNullHoacSaiKieu":[cpk_targets["CPKTruoc"].isna().sum()],
    "SoCPKNhieu":[cpk_targets["CPKTruoc"].eq(cpk_nhieu).sum()],
    "SoOCanDien":[len(cpk_targets)],
    "SoODaDien":[nhat_ky_cpk["CPKDien"].notna().sum()],
    "SoCPKThieuSauDien":[cpk_work[cot_cpk].isna().sum()]})
tong_ket_cpk

,SoHoSo,SoNullHoacSaiKieu,SoCPKNhieu,SoOCanDien,SoODaDien,SoCPKThieuSauDien
0,299,20,1,21,21,0


## 4. Nhật ký 21 ô CPK
Chụp cả bảng để thấy CPK trước xử lý, tuổi, nhóm tuổi, anaemia, lý do, cấp nhóm, số mẫu cho, median và CPK điền. MaHoSo là số thứ tự nội bộ, có thể khác SQL.

In [7]:
nhat_ky_cpk

,MaHoSo,CPKTruoc,age,age_group,anaemia,LyDo,CapNhom,SoMauCho,MedianNhom,CPKDien
0,3,None,65.000000,A3,0.000000,NULL hoặc sai kiểu,1,40,267.000000,267
1,16,99999.000000,82.000000,A4,1.000000,CPK bị làm nhiễu,1,24,160.000000,160
2,72,None,58.000000,A2,0.000000,NULL hoặc sai kiểu,1,41,318.000000,318
3,78,None,60.8,A3,0.000000,NULL hoặc sai kiểu,1,40,267.000000,267
4,102,None,75.000000,A4,0.000000,NULL hoặc sai kiểu,1,41,364.000000,364
5,110,None,60.8,A3,0.000000,NULL hoặc sai kiểu,1,40,267.000000,267
6,112,None,55.000000,A2,0.000000,NULL hoặc sai kiểu,1,41,318.000000,318
7,133,None,46.000000,A1,0.000000,NULL hoặc sai kiểu,1,27,582.000000,582
8,138,None,68.000000,A3,1.000000,NULL hoặc sai kiểu,1,41,156.000000,156
9,148,None,60.0,A3,0.000000,NULL hoặc sai kiểu,1,40,267.000000,267


## 5. Lưu kết quả riêng N03
Lưu bảng tổng kết, nhật ký và bản làm việc trung gian sau CPK. Chưa tạo dữ liệu after cuối.

In [8]:
thu_muc_cpk = thu_muc / "ket_qua" / "07_dien_cpk"
thu_muc_cpk.mkdir(parents=True, exist_ok=True)
tong_ket_cpk.to_csv(thu_muc_cpk / "tong_ket_python.csv", index=False, encoding="utf-8-sig")
nhat_ky_cpk.to_csv(thu_muc_cpk / "nhat_ky_python.csv", index=False, encoding="utf-8-sig")
cpk_work.to_csv(thu_muc_cpk / "ban_lam_viec_sau_cpk_python.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"TrangThai":["Đã lưu kết quả riêng N03; CSV before được giữ nguyên."]})

,TrangThai
0,Đã lưu kết quả riêng N03; CSV before được giữ ...
